# TuringBench DeBERTa-v3-large Fine-tuning
Fine-tune `microsoft/deberta-v3-large` on the full TuringBench train split and evaluate on the validation split. This version is tuned to finish inside Kaggle's GPU runtime limit.

In [ ]:
!pip install -q torch==2.4.1+cu118 torchvision==0.19.1+cu118 torchaudio==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate scikit-learn pandas joblib

In [ ]:
import os

# Read HuggingFace token from Kaggle Secrets so it is never hardcoded.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print('Could not load HF_TOKEN from Kaggle Secrets:', e)
    print('Set HF_TOKEN manually or add it via Add-ons -> Secrets.')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/kaggle/working/ai-detection-at-scale'
if not os.path.exists(repo_dir):
    !git clone $repo_url $repo_dir
else:
    !git -C $repo_dir pull origin main
print('Repo ready at', repo_dir)

In [ ]:
script = os.path.join(repo_dir, 'scripts', '33_finetune_turingbench.py')
out_dir = '/kaggle/working/models/turingbench_deberta_v3_large'
os.makedirs(out_dir, exist_ok=True)

hub_model_id = 'vedangvatsa/vedang-turingbench-deberta-v3-large'

# Resume from the latest Hub checkpoint if one exists and HF_TOKEN is set.
resume_from = None
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import list_repo_refs
        refs = list(list_repo_refs(repo_id=hub_model_id).branches)
        if refs:
            resume_from = hub_model_id
            print('Resuming from Hub checkpoint:', resume_from)
    except Exception as e:
        print('No existing Hub checkpoint found, starting fresh:', e)
else:
    print('HF_TOKEN not set; checkpoint resume disabled.')

# DeBERTa-v3 with FP16 on P100 triggers "Attempting to unscale FP16 gradients".
# Use FP32 with a smaller batch size and higher gradient accumulation for safety.
cmd = (
    f"python {script} "
    f"--model_name microsoft/deberta-v3-large "
    f"--output_dir {out_dir} "
    f"--max_length 256 "
    f"--epochs 1 "
    f"--batch_size 8 "
    f"--gradient_accumulation_steps 4 "
    f"--learning_rate 2e-5 "
    f"--no_fp16 "
    f"--hub_model_id {hub_model_id} "
)
if resume_from:
    cmd += f"--resume_from_checkpoint {resume_from} "
cmd += "--seed 42"

print('Running:', cmd)
!{cmd}

In [ ]:
import shutil
results_src = os.path.join(repo_dir, 'results', 'turingbench_finetuned.csv')
results_dst = '/kaggle/working/results/turingbench_finetuned.csv'
os.makedirs('/kaggle/working/results', exist_ok=True)
if os.path.exists(results_src):
    shutil.copy(results_src, results_dst)
    print('Results copied to', results_dst)
print('Outputs in /kaggle/working/models/turingbench_deberta_v3_large')